<a href="https://colab.research.google.com/github/alexwmackay/numerai_models/blob/main/numerai_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**0.0 Import Packages**

In [1]:
!pip install scikit-learn numerapi

In [2]:
import pandas as pd
import numpy as np
import numerapi
import gc
import json
import os
from datetime import datetime
from pathlib import Path
import sklearn.linear_model
import matplotlib.pyplot as plt
from numerapi import NumerAPI
from google.colab import userdata
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:

def import_numerai_data(
    version,
    cache_dir="/content/drive/My Drive/numerai_data",
    downsample_every_n = 1
    ):

    # Paths
    Path(cache_dir).mkdir(parents=True, exist_ok=True)

    train_path = f"{cache_dir}/train.parquet"
    val_path = f"{cache_dir}/validation.parquet"
    features_path = f"{cache_dir}/features.json"

    # Download data
    napi = NumerAPI(
        public_id=userdata.get('NUMERAI_PUBLIC_ID'),
        secret_key=userdata.get('NUMERAI_SECRET_KEY')
        )

    napi.download_dataset(f"{version}/train.parquet", dest_path=train_path)
    napi.download_dataset(f"{version}/validation.parquet", dest_path=val_path)
    napi.download_dataset(f"{version}/features.json", dest_path=features_path)

    with open(features_path) as f:
        feature_metadata = json.load(f)
    features = feature_metadata["feature_sets"]["small"]
    cols = ["era"] + features + ["target"]

    # Load data
    print(f"Loading small features ({len(features)} features)...")
    train = pd.read_parquet(train_path, columns=cols)
    val = pd.read_parquet(val_path, columns=cols)

    # Convert to float32
    train[features] = train[features].astype('float32')
    val[features] = val[features].astype('float32')

    # Filter validation rows with targets
    if "data_type" in val.columns:
        val = val[val["data_type"] == "validation"]

    # Downsample
    if downsample_every_n > 1:
        train = train[train["era"].isin(train["era"].unique()[::downsample_every_n])]
        val = val[val["era"].isin(val["era"].unique()[::downsample_every_n])]

    print(f"✓ Train: {train.shape} | Val: {val.shape}")
    gc.collect()

    return train, val, features


**1.0 Data Download**

In [7]:
train, val, features = import_numerai_data(version = "v5.3", downsample_every_n=4)

Loading small features (42 features)...
✓ Train: (688182, 44) | Val: (1032116, 44)


**2.0 Visualise**
